In [ ]:
#imports 
import rasterio
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

## Importing data:

In [ ]:
back = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif")
bath = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif")
train_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv')

In [ ]:
print('Train set: ', train_df.shape)
display(train_df.head())

print('Test set: ', test_df.shape)
display(test_df.head())

back_data = back.read(1)
print('Backscatter data: ',back_data.shape)
print(back_data[:5, :5])

bath_data = bath.read(1)
print('Bathymestry data: ', bath_data.shape)
print(bath_data[:5, :5])

## Extracting features:

In [ ]:
def get_features(x, y):
    row, col = bath.index(x, y)
    return bath_data[row, col], back_data[row, col]

train_df[['depth', 'scatter']] = train_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

test_df[['depth', 'scatter']] = test_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

In [ ]:
train_df.info()
test_df.info()

In [ ]:
display(train_df[['depth','scatter']].describe())
display(test_df[['depth','scatter']].describe())

It can be observerd that the **min** values of test data are insider the bounds of the training set but that isnt the case for the **max** values, so the model is bound to cause some prediction errors if it heavily relies only on these two features

## Baseline model

We will be training a **LogisticRegression** model since a **RandomForest** and a **decisiontree** would be a overkill on 2 features 

In [ ]:
target = 'class'

X = train_df[["depth","scatter"]]
y = train_df[target].map({'SGAM': 0, 'NVB': 1, 'SGZ': 2, 'ALG': 3, 'FMAT': 4})

X_test = test_df[['depth','scatter']]

In [ ]:
#splitting the data into train/val splits
X_train , X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=24)

### LogisticRegression

In [ ]:
#Model
model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs'
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model)
])

pipeline.fit(X_train, y_train)

In [ ]:
val_preds = pipeline.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='weighted')

print('Baseline validation accuracy: ', val_acc)
print('Baseline validation f1: ', val_f1)

### RandomForestCLassifier
Even though our data is limited a tree based model can find good splits that can work in favor for classification tasks 

In [ ]:
model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

In [ ]:
val_preds = model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='weighted')

print('Baseline validation accuracy: ', val_acc)
print('Baseline validation f1: ', val_f1)

as can be observed a tree based model worked better than logistic regression

## Final predictions 

In [ ]:
y_preds = model.predict(X_test)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    target: y_preds
})

submission[target] = submission[target].map({0: 'SGAM', 1: 'NVB', 2: 'SGZ', 3: 'ALG', 4: 'FMAT'})

submission.to_csv('submission.csv', index=False)